# Kraken Max v3 — Walk-Forward Research

Run in **QuantConnect Research** or locally to optimize ensemble weights and validate **out-of-sample Sharpe** before live deployment.

## Steps
1. Pull hourly Kraken history for BTCUSD, ETHUSD, SOLUSD (or load CSV)
2. Run walk-forward grid on `w_momentum`, `w_breakout`, `w_dip`, `w_ml`
3. Save `ensemble_weights.json` — auto-loaded by `KrakenMaxAlgorithm` on deploy

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()  # Kraken Max folder when notebook is in research/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from workflow import walk_forward_optimize, save_ensemble_weights

In [ ]:
# Option A: QuantConnect Research — uncomment and set tickers
# qb = QuantBook()
# syms = [qb.AddCrypto(t, Resolution.Hour, Market.Kraken).Symbol for t in ['BTCUSD','ETHUSD','SOLUSD']]
# hist = qb.History(syms, datetime(2022,1,1), datetime(2025,1,1), Resolution.Hour)
# bars = hist.reset_index().rename(columns={'symbol':'symbol','time':'timestamp'})

# Option B: Local CSV
csv_path = Path('../../tests/fixtures/walk_forward_bars.csv').resolve()
bars = pd.read_csv(csv_path)
print(bars.head(), '\nrows', len(bars), 'symbols', bars['symbol'].unique())

In [ ]:
result = walk_forward_optimize(bars, n_folds=3)
out = save_ensemble_weights(result)
print('OOS Sharpe:', round(result.oos_sharpe, 3))
print('OOS MaxDD:', round(result.oos_max_drawdown, 3))
print('OOS Return:', round(result.oos_total_return, 3))
print('Best weights:', result.best_weights)
print('Saved:', out)

## Go / No-Go checklist
- OOS Sharpe > 0.5 (stretch goal for aggressive crypto)
- OOS MaxDD > -35% (within strategy halt)
- Paper trade 60+ days before $1k live
- Refresh `data/fear_greed_history.csv` via `research/fetch_external_data.py`